In [1]:
import numpy as np
import pandas as pd

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

In [3]:
def process_user(user_dir):

    user_dir = Path(user_dir)

    # --------------------------------------------------
    # 1. Read user's CSV and JSON
    # --------------------------------------------------

    csv_path = user_dir / "dataset.csv"
    json_path = user_dir / "dataset.json"

    df_csv = pd.read_csv(csv_path)

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            json_data = json.load(f)

    except UnicodeDecodeError:
        with open(json_path, "r", encoding="latin-1") as f:
            json_data = json.load(f)

    # --------------------------------------------------
    # 2. Prepare empty JSON metadata table
    # --------------------------------------------------

    json_metadata = pd.DataFrame(
        columns=[
            "screen_id",
            "gaze_timestamp",
            "gaze_panelTitle",
            "gaze_elementId"
        ]
    )

    # --------------------------------------------------
    # 3. Extract JSON screen/gaze metadata if available
    # --------------------------------------------------

    screens = json_data.get("screens", [])

    if screens:

        screen_df = pd.json_normalize(screens)

        if "screenId" in screen_df.columns:

            if "gazeData" in screen_df.columns:

                # One row per gaze observation
                screen_df = screen_df.explode(
                    "gazeData",
                    ignore_index=True
                )

                # Extract gaze-level information
                gaze_df = pd.json_normalize(
                    screen_df.pop("gazeData")
                )

                gaze_df = gaze_df.rename(
                    columns={
                        "timestamp": "gaze_timestamp",
                        "panelTitle": "gaze_panelTitle",
                        "elementId": "gaze_elementId"
                    }
                )

            else:

                # No gazeData in JSON
                gaze_df = pd.DataFrame(
                    {
                        "gaze_timestamp": [pd.NaT] * len(screen_df),
                        "gaze_panelTitle": [pd.NA] * len(screen_df),
                        "gaze_elementId": [pd.NA] * len(screen_df)
                    }
                )

            # Make sure expected columns exist
            for column in [
                "gaze_timestamp",
                "gaze_panelTitle",
                "gaze_elementId"
            ]:

                if column not in gaze_df.columns:
                    gaze_df[column] = pd.NA

            # Combine screen and gaze information
            json_flat = pd.concat(
                [
                    screen_df.reset_index(drop=True),
                    gaze_df.reset_index(drop=True)
                ],
                axis=1
            )

            # Create metadata table
            json_metadata = json_flat[
                [
                    "screenId",
                    "gaze_timestamp",
                    "gaze_panelTitle",
                    "gaze_elementId"
                ]
            ].copy()

            json_metadata = json_metadata.rename(
                columns={
                    "screenId": "screen_id"
                }
            )

    # --------------------------------------------------
    # 4. Parse timestamps
    # --------------------------------------------------

    if "gaze_timestamp" in df_csv.columns:

        df_csv["gaze_timestamp"] = pd.to_datetime(
            df_csv["gaze_timestamp"],
            utc=True,
            errors="coerce"
        )

    json_metadata["gaze_timestamp"] = pd.to_datetime(
        json_metadata["gaze_timestamp"],
        utc=True,
        errors="coerce"
    )

    # --------------------------------------------------
    # 5. Merge JSON metadata onto CSV observations
    # --------------------------------------------------

    if (
        "screen_id" in df_csv.columns
        and not json_metadata.empty
    ):

        df = df_csv.merge(
            json_metadata,
            on=["screen_id", "gaze_timestamp"],
            how="left"
        )

    else:

        df = df_csv.copy()

        df["gaze_panelTitle"] = pd.NA
        df["gaze_elementId"] = pd.NA

    # --------------------------------------------------
    # 6. Add / verify user ID
    # --------------------------------------------------

    json_user_id = json_data.get("userId")

    if "user_id" not in df.columns:
        df["user_id"] = json_user_id

    # --------------------------------------------------
    # 7. Image file paths
    # --------------------------------------------------

    if "image_file" in df.columns:

        # Keep the path exactly as stored in the CSV
        df["original_image_file"] = df["image_file"]

        # Add user ID to make image references unique
        df["image_file"] = (
            df["user_id"].astype("string")
            + "/"
            + df["original_image_file"].astype("string")
        )

    else:

        df["original_image_file"] = pd.NA
        df["image_file"] = pd.NA

    # --------------------------------------------------
    # 8. Sort chronologically
    # --------------------------------------------------

    sort_columns = [
        column
        for column in [
            "user_id",
            "activity_id",
            "task_id",
            "screen_id",
            "gaze_timestamp"
        ]
        if column in df.columns
    ]

    if sort_columns:

        df = df.sort_values(
            sort_columns
        ).reset_index(drop=True)

    # --------------------------------------------------
    # 9. Check screenshot dimensions
    # --------------------------------------------------

    if "original_image_file" in df.columns:

        image_info = []

        for image_file in (
            df["original_image_file"]
            .dropna()
            .unique()
        ):

            image_path = user_dir / image_file

            try:

                with Image.open(image_path) as img:
                    image_width, image_height = img.size

                image_info.append(
                    {
                        "original_image_file": image_file,
                        "image_width": image_width,
                        "image_height": image_height,
                        "image_valid": True
                    }
                )

            except Exception:

                image_info.append(
                    {
                        "original_image_file": image_file,
                        "image_width": np.nan,
                        "image_height": np.nan,
                        "image_valid": False
                    }
                )

        image_info = pd.DataFrame(image_info)

        if not image_info.empty:

            df = df.merge(
                image_info,
                on="original_image_file",
                how="left"
            )

        else:

            df["image_width"] = np.nan
            df["image_height"] = np.nan
            df["image_valid"] = False

    else:

        df["image_width"] = np.nan
        df["image_height"] = np.nan
        df["image_valid"] = False

    # --------------------------------------------------
    # 10. Coordinate-bound flags
    # --------------------------------------------------

    if "gaze_x" in df.columns:

        df["x_out_of_bounds"] = (
            df["image_width"].notna()
            &
            (
                (df["gaze_x"] < 0)
                |
                (df["gaze_x"] >= df["image_width"])
            )
        )

    else:

        df["x_out_of_bounds"] = False

    if "gaze_y" in df.columns:

        df["y_out_of_bounds"] = (
            df["image_height"].notna()
            &
            (
                (df["gaze_y"] < 0)
                |
                (df["gaze_y"] >= df["image_height"])
            )
        )

    else:

        df["y_out_of_bounds"] = False

    # --------------------------------------------------
    # 11. Valid gaze flag
    # --------------------------------------------------

    df["valid_gaze"] = (
        df["gaze_timestamp"].notna()
        &
        df["gaze_x"].notna()
        &
        df["gaze_y"].notna()
    )

    # --------------------------------------------------
    # 12. Data-quality flags
    # --------------------------------------------------

    df["missing_panelTitle"] = (
        df["gaze_panelTitle"].isna()
    )

    df["missing_elementId"] = (
        df["gaze_elementId"].isna()
    )

    return df

In [4]:
current_dir = Path.cwd()

user_dirs = [
    p for p in current_dir.iterdir()
    if p.is_dir()
]

In [5]:
all_user_dfs = []

for user_dir in user_dirs:
    user_df = process_user(user_dir)
    all_user_dfs.append(user_df)

all_users_dataframe = pd.concat(
    all_user_dfs,
    ignore_index=True
)

C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_22032\2543315593.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_users_dataframe = pd.concat(


In [6]:
all_users_dataframe.head(5)

,user_id,activity_id,task_id,screen_id,screen_timestamp,gaze_timestamp,gaze_x,gaze_y,scroll_x,scroll_y,...,gaze_elementId,original_image_file,image_width,image_height,image_valid,x_out_of_bounds,y_out_of_bounds,valid_gaze,missing_panelTitle,missing_elementId
0,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.595000+00:00,1032.066549,125.775233,0.0,0.0,...,7d79a855-cdcd-482e-81f8-313e25129712,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,False,False
1,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.694000+00:00,1061.684898,89.873855,0.0,0.0,...,None,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,True,True
2,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.796000+00:00,1077.704124,73.281258,0.0,0.0,...,None,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,True,True
3,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.889000+00:00,1023.644899,61.726148,0.0,0.0,...,None,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,True,True
4,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.989000+00:00,1046.656193,48.708966,0.0,0.0,...,None,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,True,True


In [7]:
len(all_users_dataframe)

213221

In [8]:
# Descriptive statistics for all_users_dataframe

stats = {
    "User folders": all_users_dataframe["user_id"].nunique(),
    "Users with gaze data": all_users_dataframe.loc[
        all_users_dataframe["gaze_timestamp"].notna(), "user_id"
    ].nunique(),
    "Activities": all_users_dataframe["activity_id"].nunique(),
    "Tasks": all_users_dataframe["task_id"].nunique(),
    "Total Screens": all_users_dataframe["screen_id"].nunique(),
    "Gaze records": len(all_users_dataframe),
    "Valid gaze records": all_users_dataframe["valid_gaze"].sum(),
    "Invalid gaze records": (~all_users_dataframe["valid_gaze"]).sum(),
    "Learning elements": all_users_dataframe["gaze_elementId"].nunique(),
    "Out-of-bounds gaze records": (
        all_users_dataframe["x_out_of_bounds"]
        | all_users_dataframe["y_out_of_bounds"]
    ).sum(),
    "Duplicate records": all_users_dataframe.duplicated().sum()
}

stats_table = pd.DataFrame(
    list(stats.items()),
    columns=["Dataset characteristic", "Value"]
)

# Format out-of-bounds value with percentage
oob_count = stats["Out-of-bounds gaze records"]
oob_percentage = oob_count / len(all_users_dataframe) * 100

stats_table.loc[
    stats_table["Dataset characteristic"] == "Out-of-bounds gaze records",
    "Value"
] = f"{oob_count:,} ({oob_percentage:.2f}%)"

# Format numeric values with commas
for i in stats_table.index:
    if i != stats_table[
        stats_table["Dataset characteristic"] == "Out-of-bounds gaze records"
    ].index[0]:
        if isinstance(stats_table.loc[i, "Value"], (int, np.integer)):
            stats_table.loc[i, "Value"] = f"{stats_table.loc[i, 'Value']:,}"

stats_table

C:\Users\Gaurish Garg\AppData\Local\Temp\ipykernel_22032\3949505199.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '24,133 (11.32%)' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stats_table.loc[


,Dataset characteristic,Value
0,User folders,35
1,Users with gaze data,35
2,Activities,1
3,Tasks,1
4,Total Screens,"1,302"
5,Gaze records,"213,221"
6,Valid gaze records,"213,179"
7,Invalid gaze records,42
8,Learning elements,18
9,Out-of-bounds gaze records,"24,133 (11.32%)"


In [9]:
gaze_data = all_users_dataframe[
    all_users_dataframe["valid_gaze"]
].copy()

gaze data is nothing but valid gaze such that it is not out of bounds and not na

In [10]:
gaze_data.head()

,user_id,activity_id,task_id,screen_id,screen_timestamp,gaze_timestamp,gaze_x,gaze_y,scroll_x,scroll_y,...,gaze_elementId,original_image_file,image_width,image_height,image_valid,x_out_of_bounds,y_out_of_bounds,valid_gaze,missing_panelTitle,missing_elementId
0,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.595000+00:00,1032.066549,125.775233,0.0,0.0,...,7d79a855-cdcd-482e-81f8-313e25129712,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,False,False
1,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.694000+00:00,1061.684898,89.873855,0.0,0.0,...,None,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,True,True
2,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.796000+00:00,1077.704124,73.281258,0.0,0.0,...,None,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,True,True
3,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.889000+00:00,1023.644899,61.726148,0.0,0.0,...,None,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,True,True
4,69142eb4410888a7b66fa988,68e48f1a803e577b1177152e,6912a59f410888a7b66ed6cf,screenshot-124f194f-6f17-4921-8a34-cf863294df43,NaN,2025-11-12 07:09:39.989000+00:00,1046.656193,48.708966,0.0,0.0,...,None,screenshots/screen_000002.png,1366.0,632.0,True,False,False,True,True,True


In [11]:
interaction_df = pd.read_csv("robo.csv")

In [12]:
interaction_clean = interaction_df[
    [
        "user",
        "timeStamp",
        "verb",
        "element",
        "elementId",
        "elementType",
        "duration",
        "attempted"
    ]
].copy()

interaction_clean = interaction_clean.rename(
    columns={
        "user": "user_id",
        "timeStamp": "interaction_timestamp"
    }
)

interaction_clean["interaction_timestamp"] = pd.to_datetime(
    interaction_clean["interaction_timestamp"],
    utc=True,
    errors="coerce"
)

In [13]:
element_lookup = (
    interaction_df[
        ["elementId", "element", "elementType"]
    ]
    .drop_duplicates(subset=["elementId"])
)

interaction_df was required just to map element with element id. nothing else

In [14]:
gaze_data = gaze_data.merge(
    element_lookup,
    left_on="gaze_elementId",
    right_on="elementId",
    how="left"
)

In [15]:
len(gaze_data)

213179

In [16]:
gaze_data["aoi"] = gaze_data["element"]

In [17]:
aoi_sequence = gaze_data[
    gaze_data["aoi"].notna()
].copy()

thus we filtered out values where element was null

In [18]:
len(aoi_sequence)

139732

In [19]:
aoi_sequence = (
    aoi_sequence
    .sort_values(
        [
            "user_id",
            "activity_id",
            "task_id",
            "screen_id",
            "gaze_timestamp"
        ]
    )
    .reset_index(drop=True)
)

In [20]:
aoi_sequence["previous_aoi"] = (
    aoi_sequence
    .groupby(
        [
            "user_id",
            "activity_id",
            "task_id",
            "screen_id"
        ]
    )["aoi"]
    .shift()
)

for that user for that activity for that task id for that screen, we created what was previous aoi

In [21]:
aoi_sequence["new_gaze_episode"] = (
    aoi_sequence["aoi"] != aoi_sequence["previous_aoi"]
)

just boolean from previous aoi to current aoi

In [22]:
aoi_sequence["gaze_episode_id"] = (
    aoi_sequence
    .groupby(
        [
            "user_id",
            "activity_id",
            "task_id",
            "screen_id"
        ]
    )["new_gaze_episode"]
    .cumsum()
)

A new gaze episode starts whenever the current AOI differs from the immediately preceding AOI within the same screen.

| Time | AOI | New episode? | Episode ID |
| ---- | --- | ------------ | ---------: |
| 10.0 | Q1  | True         |      **1** |
| 10.1 | Q1  | False        |      **1** |
| 10.2 | Q1  | False        |      **1** |
| 10.3 | Q2  | True         |      **2** |
| 10.4 | Q2  | False        |      **2** |
| 10.5 | Q1  | True         |      **3** |


In [23]:
aoi_sequence.to_csv("aoi_sequence_final.csv", index=False)  

In [24]:
gaze_episodes = (
    aoi_sequence
    .groupby(
        [
            "user_id",
            "activity_id",
            "task_id",
            "screen_id",
            "gaze_episode_id",
            "aoi",
            "elementType"
        ],
        as_index=False
    )
    .agg(
        gaze_start=("gaze_timestamp", "min"),
        gaze_end=("gaze_timestamp", "max"),
        gaze_count=("gaze_timestamp", "size")
    )
)

gaze_duration_seconds is therefore the temporal span between the first and last recorded gaze observation in that continuous AOI episode.

Gaze episode duration was calculated as the difference between the earliest and latest gaze timestamps within each continuous AOI episode.

**Valid gaze observations**

↓

**Filter to observations with known AOI**

↓

**Order chronologically by**

`user → activity → task → screen → timestamp`

↓

**`aoi_sequence`**

*(previous AOI identified using `groupby` + `shift`)*

↓

**Identify continuous AOI runs**

↓

**`gaze_episodes`**

*(one row per continuous run)*

↓

**`gaze_start`, `gaze_end`, `gaze_count`**

↓

**`gaze_duration_seconds`**

↓

**Average total gaze duration**

**per learning element**

---

## `gaze_count`

`gaze_count` is purely mechanical: it tells us how many raw gaze-observation rows were grouped into that particular gaze episode.

For example:

**Episode 1**

- AOI: Story 1
- gaze observations: 2
- → `gaze_count = 2`

It does not mean:

- 2 fixations
- 2 seconds of gaze
- 2 switches
- 2 units of attention

It is simply:

> `gaze_count = number of gaze observations belonging to the episode`

Our actual duration measure comes from:

> `gaze_end − gaze_start`

---

## Gaze Duration and Fixation Duration

It can be related to fixation duration, but we should not call our `gaze_duration_seconds` a fixation.

In our data:

> Gaze duration = `gaze_end − gaze_start` for a continuous AOI episode.

A fixation normally requires a separate fixation-detection method/algorithm that determines when gaze is sufficiently stable on a location for a specified duration.

We did not perform that fixation detection. Therefore, we use:

- “gaze episode duration” or “gaze duration”

rather than:

- ❌ “fixation duration”

For example, your 1.005 s episode means:

> The learner's recorded gaze remained within the same AOI episode from 07:09:39.595 to 07:09:40.600, giving a temporal span of 1.005 s.

It could contain one or more fixations, but our current data processing does not let us determine that.


In [25]:
gaze_episodes["gaze_duration_seconds"] = (
    gaze_episodes["gaze_end"]
    - gaze_episodes["gaze_start"]
).dt.total_seconds()

In [26]:
len(gaze_episodes)

25667

In [27]:
gaze_episodes.to_csv("gaze_episodes_final.csv", index=False)